# Lesson 5.4 — Managing Context in Large Codebase Exploration

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Recognize context degradation in extended sessions
- Use scratchpad files to persist key findings across context boundaries
- Delegate verbose exploration to subagents while the main agent coordinates
- Design crash-recovery via structured state exports (manifests)

**You will produce:** A scratchpad writer/reader, a subagent-delegation plan, and a crash-recovery manifest.

## Concepts

In extended sessions models **degrade**: they give inconsistent answers and start referencing "typical patterns" instead of the specific classes discovered earlier. Countermeasures:

- **Scratchpad files** persist key findings across context boundaries; reference them for later questions.
- **Subagent delegation** isolates verbose exploration output while the main agent keeps high-level coordination (spawn one to "find all test files" or "trace refund flow deps").
- **Summarize** key findings from one phase before spawning subagents for the next, injecting the summary into initial context.
- **Crash recovery via manifests:** each agent exports structured state to a known location; the coordinator loads a manifest on resume and injects it into agent prompts.
- Use **`/compact`** to reduce context usage when it fills with verbose discovery output.

> **Reference:** [Anthropic - Claude Code context & /compact](https://docs.anthropic.com/en/docs/claude-code)

In [ ]:
import os, json, pathlib
WS = pathlib.Path('explore_ws'); WS.mkdir(exist_ok=True)
os.chdir(WS)
print('workspace:', os.getcwd())

## Code-Along: Step 1 — Scratchpad to persist findings

Write discoveries to disk so they survive context boundaries.

In [ ]:
def scratch_write(key, finding, path='scratchpad.json'):
    data = {}
    if pathlib.Path(path).exists():
        data = json.loads(pathlib.Path(path).read_text(encoding='utf-8'))
    data[key] = finding
    pathlib.Path(path).write_text(json.dumps(data, indent=2), encoding='utf-8')

def scratch_read(path='scratchpad.json'):
    return json.loads(pathlib.Path(path).read_text(encoding='utf-8'))

scratch_write("RefundService", "handles refunds in billing/refund.py:12")
scratch_write("entry_point", "api/routes.py -> create_refund")
print(scratch_read())

## Code-Along: Step 2 — Delegate verbose exploration to a subagent

The main agent keeps a short summary; the subagent absorbs the verbose output.

In [ ]:
def delegate(question):
    return {"subagent_task": question,
            "returns": "concise summary only (verbose output stays in subagent)"}

print(delegate("find all test files"))
print(delegate("trace refund flow dependencies"))

## Code-Along: Step 3 — Crash-recovery manifest

Each agent exports state; the coordinator reloads the manifest on resume.

In [ ]:
def export_state(agent, state, path='manifest.json'):
    manifest = {}
    if pathlib.Path(path).exists():
        manifest = json.loads(pathlib.Path(path).read_text(encoding='utf-8'))
    manifest[agent] = state
    pathlib.Path(path).write_text(json.dumps(manifest, indent=2), encoding='utf-8')

def resume_from_manifest(path='manifest.json'):
    m = json.loads(pathlib.Path(path).read_text(encoding='utf-8'))
    return {a: f"inject into prompt: {s}" for a, s in m.items()}

export_state("mapper", {"phase": "structure_mapped", "files": 128})
export_state("tracer", {"phase": "refund_flow_done"})
print(resume_from_manifest())

## Exercise

Add a `/compact`-style `compact(history, keep_last=5)` that keeps a short summary plus the last N turns, and injects the scratchpad summary at the top. Then simulate a crash: write a manifest, clear memory, and assert `resume_from_manifest` restores both agents' state.

In [ ]:
# TODO: implement compact(history, keep_last=5) injecting scratch_read() summary
# TODO: simulate crash + assert resume_from_manifest() has 'mapper' and 'tracer'

## Key Takeaways

- Extended sessions degrade - models drift to 'typical patterns' over specific findings.
- Scratchpad files persist key findings across context boundaries.
- Delegate verbose exploration to subagents; the main agent keeps high-level coordination.
- Use manifests for crash recovery and /compact to reduce bloated context.

---

**Next:** [Lesson 5.5 — Human Review & Confidence Calibration](./lesson-5.5-human-review-confidence.ipynb)